# Import Libraries

In [1]:
import os 
import shutil #recursively moves a file from src to dest and vice versa
import cv2 as cv #resize image,gif
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
from tensorflow.keras import utils

# Auto Folder Creation

In [2]:
def create_dir(in_dir,out_dir,gdir_1,gdir_2):
    for label in os.listdir(path=in_dir):
        if not os.path.exists(os.path.join(out_dir,gdir_1,label)):
            os.makedirs(os.path.join(out_dir,gdir_1,label))
        if not os.path.exists(os.path.join(out_dir,gdir_2,label)):
            os.makedirs(os.path.join(out_dir,gdir_2,label))

# Gabor Filter function

In [3]:
def preprocess_image(in_dir = 'Data', out_dir= 'processed'):
    gdir_1 = 'gabor_1'
    gdir_2 = 'gabor_2'
    
    create_dir(in_dir, out_dir, gdir_1, gdir_2)
    
    for label in os.listdir(path= in_dir):
        print('Processing started for : ',label)
        for image_name in os.listdir(os.path.join(in_dir,label)):
            if not image_name.endswith('.db'):
                in_path = os.path.join(in_dir, label, image_name)
                out_path_1 = os.path.join(out_dir, gdir_1, label, image_name) #path of gabor1
                out_path_2 = os.path.join(out_dir, gdir_2, label, image_name)# path of gabor 2
#                 print('input_path : ',in_path, '\nout_path_1 : ', out_path_1, '\nout_path_2 : ', out_path_2)
                img = cv.imread(in_path,0) #reading image 
                gabor_1 = cv.getGaborKernel((18, 18), 1.5, np.pi/4, 5.0, 1.5, 0, ktype=cv.CV_32F) #initialising the parameters of gabor filter 
                filtered_img_1 = cv.filter2D(img, cv.CV_8UC3, gabor_1) # applying gabor filter
                cv.imwrite(out_path_1, filtered_img_1) #writing filtered image in out_oath
                gabor_2 = cv.getGaborKernel((18, 18), 1.5, np.pi/4, 5.0, 1.5, 0, ktype=cv.CV_32F)
                filtered_img_2 = cv.filter2D(filtered_img_1, cv.CV_8UC3, gabor_2)
                cv.imwrite(out_path_2, filtered_img_2)
        print('------ Processing done ------')

In [4]:
preprocess_image()

Processing started for :  Fake
------ Processing done ------
Processing started for :  Real
------ Processing done ------


In [5]:
input_dir=os.path.join('processed','gabor_2')
print('input_dir:',input_dir)

input_dir: processed\gabor_2


In [6]:
no_samples = 0
labels = os.listdir(input_dir)
print('class labels:',labels)

for label in labels:
    total = len(os.listdir(os.path.join(input_dir,label)))
    print(label,total)
    no_samples+=total
print('-'*30)
print('Total no of Samples:',no_samples)
print('-'*30)

class labels: ['Fake', 'Real']
Fake 960
Real 1081
------------------------------
Total no of Samples: 2041
------------------------------


In [7]:
img_rows =128
img_cols =128
channels = 3

# Converting the data into numpy files

In [8]:
def load_data(input_dir, no_samples, labels):
    
    images = np.ndarray((no_samples, img_rows, img_cols, channels), dtype = np.float32)
    targets = np.zeros((no_samples,), dtype = np.uint8)
    
    i = 0
    print('-'*30)
    print('Loading training images...')
    print('-'*30)
    
    for j, label in enumerate(labels):
        image_names = os.listdir(os.path.join(input_dir, label))
        total = len(image_names)
        print(label, total)
        for image_name in image_names:
            img = cv.imread(os.path.join(input_dir, label, image_name), 1)
            img = np.array(cv.resize(img, (img_rows, img_cols)))
            
            images[i] = np.reshape(a= img, newshape= (img_rows, img_cols, channels))
            targets[i] = j
            
            i += 1
    print('Loading done.')
    
    targets = utils.to_categorical(y= targets, num_classes= len(labels))
    
    return images, targets

In [9]:
X,Y = load_data(input_dir,no_samples,labels)

------------------------------
Loading training images...
------------------------------
Fake 960
Real 1081
Loading done.


In [11]:
print(X.shape,Y.shape)

(2041, 128, 128, 3) (2041, 2)


In [15]:
np.savez(file='dataset/data',x=X,y=Y)

# Done